In [1]:
!pip install -U transformers accelerate huggingface_hub

In [2]:
!pip install pandas

In [4]:
from transformers import pipeline
import transformers
import pandas as pd
import torch
import os

In [5]:
print(transformers.__version__)

5.22.1


In [6]:
from huggingface_hub import login
login()

In [7]:

# ---- Modelo ----
pipe = pipeline(
    "image-text-to-text",
    model="google/translategemma-4b-it",
    device="cuda",
    dtype=torch.bfloat16
)
pipe.processor.tokenizer.padding_side = "left"  # necesario para traducir en lotes
 
 
def armar_mensaje(texto, destino):
    return [{
        "role": "user",
        "content": [{
            "type": "text",
            "source_lang_code": "en",
            "target_lang_code": destino,
            "text": str(texto),
        }],
    }]
 
 
def traducir_csv(entrada, salida, destino, tam_lote=8, prueba=None):
    """
    Lee el CSV en inglés, agrega una columna nueva con la traducción
    y deja el texto original como está.
    """
    df = pd.read_csv(entrada)
    if prueba:
        df = df.head(prueba)  # para probar con pocas filas
 
    col_nueva = f"Review Text_{destino}"
    textos = df["Review Text"].tolist()
 
    # si ya había avance guardado, sigue desde ahí
    hechas = []
    if os.path.exists(salida):
        hechas = pd.read_csv(salida)[col_nueva].tolist()
    print(f"Ya traducidas: {len(hechas)} de {len(textos)}")
 
    for i in range(len(hechas), len(textos), tam_lote):
        lote = [armar_mensaje(t, destino) for t in textos[i:i + tam_lote]]
        out = pipe(text=lote, max_new_tokens=1024, batch_size=tam_lote)
        hechas += [o[0]["generated_text"][-1]["content"] for o in out]
 
        # guarda cada ~200 reseñas y al final
        if len(hechas) % 200 < tam_lote or len(hechas) == len(textos):
            parcial = df.iloc[:len(hechas)].copy()
            parcial[col_nueva] = hechas
            parcial.to_csv(salida, index=False)
            print(f"{len(hechas)} / {len(textos)}")
 
    print("Listo:", salida)
    return pd.read_csv(salida)

# pt = traducir_csv("resenas_portugues.csv", "resenas_portugues_trad.csv", "pt-BR")
# es = traducir_csv("resenas_espanol.csv", "resenas_espanol_trad.csv", "es")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [8]:
#PRUEBA
p = traducir_csv("resenas_portugues.csv", "prueba_pt.csv", "pt-BR", prueba=10)
print(p[["Review Text", "Review Text_pt-BR"]])

Ya traducidas: 10 de 10
Listo: prueba_pt.csv
                                         Review Text  \
0  You can't go wrong here. There are a ton of ch...   
1  One of the best I e cream shops I've ever stop...   
2  Stumptown coffee, amazing gigantic donuts...ge...   
3  Best fresh frozen custard. Even the most basic...   
4  I finally came early enough to get a French to...   
5  The bakery is legit... that's all I really hav...   
6  Very unique as you walk in the front and exit ...   
7  Lots of variety in pastries which was nice. To...   
8  The cupcakes are amazing! I always stop to bri...   
9  Probably my favorite place in Madison during o...   

                                   Review Text_pt-BR  
0  Aqui, você não pode errar. Há uma grande varie...  
1  Um dos melhores sorveterias que já visitei! El...  
2  Café Stumptown, donuts gigantes e deliciosos.....  
3  A melhor sobremesa de creme congelado. Mesmo o...  
4  Finalmente, cheguei a tempo de conseguir um do...  
5  A pad

In [9]:
pd.set_option("display.max_colwidth", None)

In [10]:
p

,Rating,Date,Review Text,Review Text_pt-BR
0,5,6/3/2017,You can't go wrong here. There are a ton of choices and a scoop is pretty big it's a tough decision. They are very helpful with letting you taste anything you want. I did the blueberry cobbler and black cherry delicious!! Next time I'm going for the coffee chip my friend got it and it was amazing!!,"Aqui, você não pode errar. Há uma grande variedade de opções e as porções são bem generosas, o que torna a escolha difícil. Eles são muito prestativos e permitem que você experimente qualquer sabor que desejar. Eu experimentei o ""cobbler"" de mirtilo e cereja preta, e estava delicioso!! Na próxima, vou experimentar o ""coffee chip"", meu amigo experimentou e adorou!"
1,5,2/18/2016,"One of the best I e cream shops I've ever stopped at! They insist you try as many flavors as possible, and you should, and each option is somehow better than the last. The staff is ridiculously nice, super clean inside, and several amazing sorbets for dairy free options. I'm excited to go back again, and again.","Um dos melhores sorveterias que já visitei! Eles incentivam você a experimentar o máximo de sabores possível, e você deve, pois cada opção é, de alguma forma, melhor que a anterior. A equipe é incrivelmente atenciosa, o ambiente é impecável e eles oferecem vários sorvetes deliciosos, incluindo opções sem lactose. Estou muito animado para voltar novamente."
2,5,5/1/2016,"Stumptown coffee, amazing gigantic donuts...get here early or they will run out of your favorites. Free parking around the corner in the lot by the juice store. Met the owner and founder ...everyone in there is so nice. Will have to be back soon too many good choices.","Café Stumptown, donuts gigantes e deliciosos... chegue cedo, pois os seus sabores preferidos podem acabar rapidamente. Estacionamento gratuito na rua ao lado, no estacionamento próximo à loja de sucos. Conheci o proprietário e o fundador... todos são muito simpáticos. Tenho que voltar em breve, pois há muitas opções ótimas."
3,5,5/3/2021,"Best fresh frozen custard. Even the most basic of flavors are worth getting. It's definitely not like any place nearby with ice cream. The custard is actually somewhat addicting, and will keep you and your family coming back for more.","A melhor sobremesa de creme congelado. Mesmo os sabores mais simples valem a pena experimentar. Definitivamente, não se compara a nenhum outro lugar por perto que venda sorvete. O creme é realmente delicioso, e certamente fará você e sua família voltarem sempre."
4,4,10/12/2016,"I finally came early enough to get a French toast donut! I had been teased for weeks by this donut, from Instagram and yelp! I tried the Sunday before last and missed out as I came too late, surprisingly 9:40 is too late even though they open at 8! This place is very popular. Unfortunately, you have to arrive super early to get a FT donut which they only serve on the weekends. I arrived last Saturday before the doors opened and it was well worth it! The donut was so soft and tasty, it went down smooth with the milk I ordered. I have to say it sucked sitting at the eating area as the air is blasting which made my French toast cool after a while. I only ordered the French Toast so I'm not sure about the rest of the donuts. I do plan on going back this weekend for another FT donut so maybe I'll try some others as well. The 4 stars is because I feel they should have the FT donut all week and because the hours they are open is so short, some days they close at noon. They could be making so much more business!","Finalmente, cheguei a tempo de conseguir um donut de ""French toast""! Estava ansioso por este donut há semanas, depois de ver posts no Instagram e em avaliações online. Tentei no domingo anterior e não consegui, pois cheguei tarde demais – para minha surpresa, 9h40 já é tarde, mesmo com a loja abrindo às 8! Este lugar é muito popular. Infelizmente, é preciso chegar muito cedo para conseguir um donut de ""French to

In [11]:
pt = traducir_csv("resenas_portugues.csv", "resenas_portugues_trad.csv", "pt-BR")


Ya traducidas: 5000 de 6632


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


5200 / 6632
5400 / 6632
5600 / 6632
5800 / 6632
6000 / 6632
6200 / 6632
6400 / 6632
6600 / 6632
6632 / 6632
Listo: resenas_portugues_trad.csv


In [ ]:
es = traducir_csv("resenas_espanol.csv", "resenas_espanol_trad.csv", "es")